In [1]:
!nvidia-smi -L || echo "No GPU"
!python -V
%pip -q install -U transformers accelerate bitsandbytes

import torch, transformers
if not torch.cuda.is_available():
    raise RuntimeError("Please enable the GPU runtime.")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-81b957b9-9448-32fd-2494-abcd1383fb6f)
Python 3.13.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.5 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
Transformers: 5.17.0
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB


In [2]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True
DISABLE_XET = False
OFFLINE_MODE = False

CACHE_DIR = PROJECT_DIR/"Program"/"hf_cache_qwen38_27b" if USE_DRIVE_CACHE else Path("/content/hf_cache_qwen38_27b")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)
if DISABLE_XET: os.environ["HF_HUB_DISABLE_XET"]="1"
if OFFLINE_MODE: os.environ["HF_HUB_OFFLINE"]="1"

usage=shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print(f"free space: {usage.free/1024**3:.1f} GB")
if USE_DRIVE_CACHE and usage.free < 70*1024**3:
    print("WARNING: 70GB程度以上の空き容量を推奨します。")


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache_qwen38_27b
free space: 178.6 GB


- 時間がかかります。

In [13]:
# ===========================================================================
# Codice 3: Modello Qwen3.8-27B (Versione Uncensored Corretta & Unblithered)
# ===========================================================================
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# Sostituito con l'indice completo funzionante senza file safetensors mancanti
MODEL_ID = "wangzhang/Qwen3.8-27B-abliterated"

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    cache_dir=str(CACHE_DIR),
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device


@torch.inference_mode()
def chat_generate(
    messages,
    max_new_tokens=4096,  # Spazio generoso obbligatorio per i token di pensiero
    do_sample=True,
    temperature=1.0,
    top_p=0.95,
):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,

        # Attiviamo l'unblithered mode (Deep Thinking ragionamento nativo)
        enable_thinking=True,

        return_dict=True,
        return_tensors="pt",
    ).to(INPUT_DEVICE)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "repetition_penalty": 1.0,
        "use_cache": True,
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = float(top_p)
        generation_kwargs["top_k"] = 20

    outputs = model.generate(
        **generation_kwargs
    )

    generated_ids = outputs[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer


print("Modello Uncensored caricato correttamente:", MODEL_ID)
print("Dispositivo di input:", INPUT_DEVICE)
print("Caricato in 4bit:", getattr(model, "is_loaded_in_4bit", False))
print("Dtype di calcolo:", COMPUTE_DTYPE)


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Modello Uncensored caricato correttamente: wangzhang/Qwen3.8-27B-abliterated
Dispositivo di input: cuda:0
Caricato in 4bit: True
Dtype di calcolo: torch.bfloat16


In [11]:
# =========================================
# コード4 動作確認
# =========================================
messages=[
  {"role":"system","content":"Sei un assistente utile che risponde in modo conciso in italiano."},
  {"role":"user","content":"Per favore, presentati in una sola frase in italiano."},
]
print(chat_generate(messages,max_new_tokens=96,do_sample=False))



We need answer user: "Per favore, presentati in una sola frase in italiano." Need produce one sentence in Italian. Concise. Maybe "Sono un assistente AI utile che risponde in modo conciso in italiano." Ensure one sentence. Final only that.
</think>

Sono un assistente AI utile che risponde in modo conciso in italiano.


In [6]:
# =========================================
# コード5 日本語推論テスト
# =========================================
messages=[
  {"role":"system","content":"Sei un assistente che risponde in modo chiaro in italiano."},
  {"role":"user","content":"Acquisto un libro con un prezzo di listino di 3000 yen con uno sconto del 20%, e su questo prezzo si applica un'imposta sui consumi del 10%. Per favore, spiega brevemente l'importo da pagare insieme al processo di calcolo."},
]
print(chat_generate(messages,max_new_tokens=256,do_sample=False))



Ecco il calcolo passo dopo passo:

1.  **Prezzo scontato**: Il prezzo di listino è 3000 yen. Con uno sconto del 20%, il nuovo prezzo è $3000 \times (1 - 0,20) = 3000 \times 0,80 = 2400$ yen.
2.  **Aggiunta dell'imposta**: Su questo prezzo si applica un'imposta del 10%. L'importo dell'imposta è $2400 \times 0,10 = 240$ yen.
3.  **Totale da pagare**: Sommando il prezzo scontato e l'imposta, otteniamo $2400 + 240 = 2640$ yen.

L'importo totale da pagare è **2640 yen**.


In [14]:
# ===========================================================================
# Codice 6: Interfaccia Chat LLM Locale Gradio (Supporto Codice Lungo)
# ===========================================================================
import gradio as gr
import inspect, torch

def make_messages_chatbot(**kwargs):
    if "type" in inspect.signature(gr.Chatbot).parameters:
        kwargs["type"] = "messages"
    return gr.Chatbot(**kwargs)

print("Versione di Gradio:", gr.__version__)

def gr_chat(history, user_msg):
    history = history or []
    user_msg = (user_msg or "").strip()
    if not user_msg:
        return history, "", history
    messages = history[-8:] + [{"role": "user", "content": user_msg}]
    try:
        # Aumentato max_new_tokens a 4096 per non tagliare le risposte con codice lungo
        reply = chat_generate(messages, max_new_tokens=4096, do_sample=False)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        reply = "Si è verificato un errore 'CUDA out of memory'. Cancella la cronologia della chat (Clear) o riavvia l'ambiente di runtime."
    except Exception as e:
        reply = f"{type(e).__name__}: {e}"
    new_history = history + [{"role": "user", "content": user_msg}, {"role": "assistant", "content": reply}]
    return new_history, "", new_history

with gr.Blocks(title="Qwen3.8-27B Local Chat") as chat_demo:
    gr.Markdown("## Qwen3.8-27B — Chat Locale in 4-bit")

    # Abilitato il rendering Markdown completo per formattare i blocchi di codice lunghi
    chatbot = make_messages_chatbot(label="Chat", show_label=False, sanitize_html=True, render_markdown=True)
    chat_state = gr.State([])

    # Modificato in una casella multilinea (lines=5) per permetterti di incollare codice lungo in input
    user_box = gr.Textbox(placeholder="Inserisci qui la tua domanda o incolla il tuo codice...", label="", lines=5, max_lines=20)

    with gr.Row():
        send_btn = gr.Button("Invia", variant="primary")
        clear_btn = gr.Button("Cancella")

    send_btn.click(gr_chat, inputs=[chat_state, user_box], outputs=[chat_state, user_box, chatbot], queue=False)
    clear_btn.click(lambda: ([], "", []), outputs=[chat_state, user_box, chatbot])

print("AVVISO: Con 'share=True' verrà creato un link pubblico condivisibile.")
chat_demo.launch(share=True, inline=True, debug=False)


Versione di Gradio: 6.26.0
AVVISO: Con 'share=True' verrà creato un link pubblico condivisibile.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://704c7d4b9f40b0a091.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# =========================================
# コード7 GPUメモリ使用量を確認
# =========================================
import torch
print("GPU allocated: %.2f GB"%(torch.cuda.memory_allocated()/1024**3))
print("GPU reserved : %.2f GB"%(torch.cuda.memory_reserved()/1024**3))
free,total=torch.cuda.mem_get_info()
print("GPU free      : %.2f GB"%(free/1024**3))
print("GPU total     : %.2f GB"%(total/1024**3))


In [ ]:
# =========================================
# コード8 bitsandbytesの4bit量子化設定を確認
# =========================================
print(bnb_config)
